<a href="https://colab.research.google.com/github/tuan314159265/parking-occupancy/blob/main/notebooks/01_pklot_run_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nhận diện trạng thái ô đỗ xe — PKLot


## 1. Bài toán và mục tiêu


<!-- TODO -->


## 2. Môi trường và cấu hình


In [31]:
OWNER = "tuan314159265"
REPO = "parking-occupancy"
REF = "main"
MODULE_PATH = "parking_module.py"

from pathlib import Path
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "parking-occupancy"  # Thư mục CÁ NHÂN để lưu kết quả/module.
DOWNLOAD_DATA = True
FORCE_AUDIT = False  # True nếu dataset đã thay đổi hoặc muốn kiểm tra lại toàn bộ.

# Để trống: tự tìm/tải dataset trong thư mục cá nhân.
# Dùng chung: thêm shortcut Drive, rồi điền thư mục chứa camera UFPR04/UFPR05/PUCPR.
SHARED_DATASET_DIR = ""  # Ví dụ /content/drive/MyDrive/PKLot-shared/data/raw/PKLot
# Không bắt buộc: thư mục outputs/full của chủ dataset (chỉ đọc).
SHARED_AUDIT_DIR = ""  # Ví dụ /content/drive/MyDrive/PKLot-shared/outputs/full

# Bộ manifest dùng chung, đã khóa trên GitHub.
PROJECT_DATASET_VERSION = "pklot_v1"
PROTOCOL = "within_camera"  # Hoặc transfer_ufpr04_to_ufpr05; phải huấn luyện lại trên split của giao thức đó.


### Kết nối Google Drive

Trong cửa sổ cấp quyền Google, chọn tài khoản Drive của bạn, đánh dấu **tất cả các quyền được yêu cầu** (hoặc **Select all**) rồi bấm **Continue**. Bỏ sót một quyền có thể gây lỗi `credential propagation was unsuccessful`. Không chỉ chọn riêng quyền đọc/ghi Drive.

Nếu không thấy cửa sổ, kiểm tra popup bị chặn. Nếu vẫn lỗi dù đã cấp đủ quyền, thử Runtime → Disconnect and delete runtime rồi kết nối lại. Thao tác này xóa dữ liệu tạm của runtime, giữ file trên Drive. Nếu dùng local runtime, đặt USE_GOOGLE_DRIVE=False.

Nguồn: https://github.com/googlecolab/colabtools/issues/6089


In [32]:
import os
from pathlib import Path

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        from google.colab._message import MessageError
    except ImportError as exc:
        raise RuntimeError(
            "Kết nối bằng runtime Google Colab để dùng Drive. "
            "Nếu chạy local, đặt USE_GOOGLE_DRIVE=False."
        ) from exc

    mountpoint = Path("/content/drive")
    def drive_ready():
        return os.path.ismount(mountpoint) and (mountpoint / "MyDrive").is_dir()

    if not drive_ready():
        for attempt in range(2):
            try:
                print("Kết nối Google Drive: chọn tài khoản, chọn TẤT CẢ quyền được yêu cầu rồi Continue.")
                drive.mount(str(mountpoint), force_remount=(attempt == 1), timeout_ms=120000)
                if not drive_ready():
                    raise RuntimeError("Drive chưa có thư mục MyDrive sau khi mount.")
                break
            except MessageError as exc:
                raise RuntimeError(
                    "Colab chưa nhận được thông tin xác thực Google (MessageError). "
                    "Chạy lại cell này; trong hộp thoại đăng nhập, chọn đúng tài khoản "
                    "và đánh dấu tất cả quyền Drive/Colab được yêu cầu trước khi Continue. "
                    "Nếu không thấy hộp thoại, kiểm tra trình duyệt có chặn popup. "
                    "Nếu vẫn lỗi, dùng Runtime > Disconnect and delete runtime rồi kết nối lại. "
                    "Chưa tải dataset và chưa chuyển sang bộ nhớ tạm. "
                    f"Chi tiết gốc: {exc}"
                ) from exc
            except (ValueError, OSError, RuntimeError) as exc:
                if attempt == 0:
                    print("Kết nối chưa thành công; thử lại bằng force_remount=True.")
                else:
                    raise RuntimeError(
                        "Không kết nối được Google Drive sau 2 lần thử. "
                        "Nếu dùng local runtime, chuyển sang runtime Google Colab. "
                        "Trên Colab: chọn Runtime > Disconnect and delete runtime, "
                        "kết nối lại rồi Run All và cấp quyền Drive. "
                        "Để chạy tạm không dùng Drive, đặt USE_GOOGLE_DRIVE=False; "
                        "khi đó dữ liệu chỉ nằm trong phiên Colab. "
                        "Chưa tải dataset vì Drive chưa sẵn sàng."
                    ) from exc
    WORK_DIR = mountpoint / "MyDrive" / DRIVE_PROJECT_DIR
    print("Dữ liệu và kết quả sẽ được lưu trên Google Drive.")
else:
    WORK_DIR = Path("/content/parking-occupancy") if Path("/content").is_dir() else Path.cwd()
    print("Chế độ không dùng Drive. Trên Colab, dữ liệu sẽ mất khi runtime bị xóa.")

RAW_DIR = WORK_DIR / "data/raw"
DATASET_DIR = Path(SHARED_DATASET_DIR) if SHARED_DATASET_DIR else RAW_DIR / "PKLot"
OUTPUT_DIR = WORK_DIR / "outputs/full"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Thư mục lưu:", WORK_DIR)


Dữ liệu và kết quả sẽ được lưu trên Google Drive.
Thư mục lưu: /content/drive/MyDrive/parking-occupancy


In [33]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "Pillow>=10,<13"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'Pillow>=10,<13'], returncode=0)

In [34]:
import hashlib, importlib.util, json, sys, time, urllib.request
from urllib.parse import quote
assert OWNER != "YOUR_GITHUB_USERNAME", "Điền OWNER, REPO và REF ở cell cấu hình"
WORK_DIR.mkdir(parents=True, exist_ok=True)
url = (f"https://raw.githubusercontent.com/{quote(OWNER, safe='')}/"
       f"{quote(REPO, safe='')}/{quote(REF, safe='')}/{quote(MODULE_PATH, safe='/')}")
# Tải lại module mỗi lần Run All để lấy đúng REF đã cấu hình.
request = urllib.request.Request(
    f"{url}?refresh={time.time_ns()}", headers={"Cache-Control": "no-cache"}
)
with urllib.request.urlopen(request, timeout=60) as response:
    code = response.read()
compile(code, MODULE_PATH, "exec")
module_file = WORK_DIR / "parking_module.py"
module_file.write_bytes(code)
spec = importlib.util.spec_from_file_location("parking_module", module_file)
parking = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = parking
spec.loader.exec_module(parking)
provenance = {"url": url, "ref": REF, "sha256": hashlib.sha256(code).hexdigest()}
(WORK_DIR / "module_source.json").write_text(json.dumps(provenance, indent=2))
print("Loaded module:", provenance)


Loaded module: {'url': 'https://raw.githubusercontent.com/tuan314159265/parking-occupancy/main/parking_module.py', 'ref': 'main', 'sha256': '1ec46218d76911f3043f3f3f3e33210ea99270f03e6b56d295c1829f8290cfa4'}


### Giao diện cho các bước tiếp theo

Gọi trực tiếp `parking.VehicleDetector`, `parking.OccupancyPrediction`, `parking.occupancy_metrics` khi triển khai detector và ghép ô. Polygon XML là vùng ô đỗ, không phải hộp xe. Chưa triển khai mô hình trong notebook này.


## 3. Dữ liệu và tiền xử lý


### 3.1. Tải và kiểm tra dữ liệu PKLot


#### Dùng lại dataset đã có

Có dataset XML: bỏ qua tải/giải nén, kể cả thiếu provenance.json. Có audit tương thích: dùng lại manifest và kiểm tra nhanh một mẫu, bỏ qua audit toàn bộ. Thiếu cache: audit một lần và lưu vào thư mục cá nhân. FORCE_AUDIT=True để quét lại sau khi dữ liệu thay đổi. Cache không chứng minh dataset chưa thay đổi; chỉ dùng lại khi bộ dữ liệu được giữ nguyên.

Với dữ liệu chung, điền SHARED_DATASET_DIR và tùy chọn SHARED_AUDIT_DIR. Mã chỉ đọc các thư mục này; không tải hay giải nén vào đó. Nếu đường dẫn chung sai thì dừng để bạn sửa. DRIVE_PROJECT_DIR phải là thư mục cá nhân; không trỏ nó vào shortcut chung. Archive khoảng 4,9 GB; lần đầu cần khoảng 10–15 GB trống nếu tự tải.


In [35]:
summary = parking.ensure_dataset(
    raw=RAW_DIR,
    output=OUTPUT_DIR,
    dataset_root=DATASET_DIR if SHARED_DATASET_DIR else None,
    cached_audit_dir=SHARED_AUDIT_DIR or None,
    download=DOWNLOAD_DATA,
    force_audit=FORCE_AUDIT,
)
print(json.dumps(summary, indent=2))
assert summary["valid_frames"] > 0


Existing dataset: /content/drive/MyDrive/parking-occupancy/data/raw/PKLot. Skip download and extraction.
No compatible audit cache or FORCE_AUDIT=True; audit existing dataset once.
{
  "audit_version": 2,
  "root": "/content/drive/MyDrive/parking-occupancy/data/raw/PKLot",
  "seed": 42,
  "xml_count": 12416,
  "valid_frames": 11638,
  "rejected_frames": 778,
  "images_without_xml": 1,
  "counts": {
    "camera/PUCPR": 4077,
    "weather/overcast": 4097,
    "split/train": 9080,
    "vacant": 348315,
    "occupied": 319261,
    "date_directory_mismatch": 195,
    "weather/rainy": 1206,
    "split/val": 1626,
    "weather/sunny": 6335,
    "split/test": 932,
    "camera/UFPR04": 3547,
    "camera/UFPR05": 4014
  }
}


In [36]:
# Đọc một mẫu hợp lệ trực tiếp bằng module lấy từ GitHub.
with (OUTPUT_DIR / "valid_samples.jsonl").open() as stream:
    sample = json.loads(next(stream))
frame = parking.read_frame(DATASET_DIR / sample["xml_path"], DATASET_DIR)
print("Sample:", frame.sample_id)
print("Camera / weather / date:", frame.camera, frame.weather, frame.date)
print("Parking spaces:", len(frame.spaces))
print("Outputs:", OUTPUT_DIR)


Sample: PUCPR/Cloudy/2012-09-12/2012-09-12_06_05_16
Camera / weather / date: PUCPR overcast 2012-09-12
Parking spaces: 100
Outputs: /content/drive/MyDrive/parking-occupancy/outputs/full


### 3.2. Chia tập dữ liệu và cấu hình vùng ô đỗ

Dùng bộ `pklot_v1` đã khóa, không chia lại ngẫu nhiên trên từng máy. Cùng ngày chỉ thuộc một split. `within_camera` dùng cả ba camera; `transfer_ufpr04_to_ufpr05` giữ riêng UFPR05 để kiểm thử góc nhìn mới cùng bãi. Chuyển giao thức phải huấn luyện lại, không dùng model đã học UFPR05.

`TRAIN_FRAMES`, `VAL_FRAMES`, `TEST_FRAMES` dùng chung cho mọi phương pháp; mỗi dòng có nhãn ô và `geometry_id` liên kết tới `SLOT_GEOMETRIES`. Ngày lấy từ tên ảnh, giữ ngày thư mục để truy vết. Nhãn `split` trong audit chỉ là bản xem trước, không thay thế các manifest đã khóa này.


In [37]:
PROJECT_SPLIT_DIR = WORK_DIR / "splits" / PROJECT_DATASET_VERSION
project_data = parking.fetch_project_splits(
    OWNER, REPO, REF, PROJECT_SPLIT_DIR, dataset_version=PROJECT_DATASET_VERSION
)
SLOT_GEOMETRIES = project_data["geometries"]
SPLIT_LOCK = project_data["lock"]
MANIFEST_DIR = PROJECT_SPLIT_DIR / PROTOCOL
TRAIN_FRAMES = project_data["protocols"][PROTOCOL]["train"]
VAL_FRAMES = project_data["protocols"][PROTOCOL]["val"]
TEST_FRAMES = project_data["protocols"][PROTOCOL]["test"]
print("Dataset / protocol:", PROJECT_DATASET_VERSION, PROTOCOL)
print("Train / validation / test:", len(TRAIN_FRAMES), len(VAL_FRAMES), len(TEST_FRAMES))
print("Geometry layouts:", len(SLOT_GEOMETRIES))
print("Manifest directory:", MANIFEST_DIR)
# Kiểm tra một frame thật; các phương pháp đọc frame theo cùng manifest này.
example_frame = parking.read_project_frame(TRAIN_FRAMES[0], DATASET_DIR)
print("Verified sample:", example_frame.sample_id)


Dataset / protocol: pklot_v1 within_camera
Train / validation / test: 1182 270 270
Geometry layouts: 5
Manifest directory: /content/drive/MyDrive/parking-occupancy/splits/pklot_v1/within_camera
Verified sample: PUCPR/Cloudy/2012-09-12/2012-09-12_06_10_30


## 4. Các phương pháp nhận diện


<!-- TODO -->


### 4.1. HOG và SVM


In [38]:
# TODO


### 4.2. Phân tích hình học bằng Canny và Hough


In [39]:
# TODO


### 4.3. ROI và chia tile


In [40]:
# TODO


### 4.4. YOLO trên toàn ảnh


In [41]:
# TODO


### 4.5. Kết hợp ROI với YOLO


In [42]:
# TODO


### 4.6. Ghép xe với ô đỗ và suy luận trạng thái


In [43]:
# TODO


## 5. Pipeline và trực quan hóa


In [44]:
# TODO


<!-- TODO -->


## 6. Đánh giá và thực nghiệm


### 6.1. Chỉ số và bộ đánh giá chung


In [45]:
# TODO


### 6.2. Validation và lựa chọn tham số


In [46]:
# TODO


### 6.3. Kiểm thử trên tập test


In [47]:
# TODO


## 7. Kết quả và thảo luận


<!-- TODO -->


## 8. Kết luận


<!-- TODO -->


## 9. Hướng dẫn tái lập


<!-- TODO -->


In [48]:
# TODO
